##  **Incremental Data Ingestion**


In bronze layer raw data is stored in Volumes because you just need a landing zone / data lake folder to dump data in. No real transformations, just “land and store.”

In [0]:
# Tell Spark to read streaming data using Auto Loader (cloudFiles)
df = (spark.readStream.format('cloudfiles')
    # The incoming files are in CSV format
    .option('cloudfiles.format', 'csv')
    # Store and track the data schema in this checkpoint folder so Spark knows column structure
    .option('cloudfiles.schemaLocation', '/Volumes/workspace/bronze/bronzevolume/bookings/checkpoint')
    # If new or unexpected columns appear, put them into a special _rescued_data column instead of failing
    .option('cloudfiles.schemaEvolutionMode', 'rescue')
    # Start reading new files as they arrive from this raw cloud storage folder
    .load('/Volumes/workspace/raw/rawvolume/rawdata/bookings/'))
   

In [0]:
# Write the streaming data out in Delta Lake format
(df.writeStream.format('delta')
    # Add new rows to the table without modifying existing data (append-only mode)
    .outputMode('append')
    # Process all available new files once and then stop
    .trigger(once=True)
    # Save the processed Delta data into this bronze storage path
    .option('path', '/Volumes/workspace/bronze/bronzevolume/bookings/data')
    # Keep track of which files have already been processed in this checkpoint folder (to avoid duplicates and resume on failure)
    .option('checkpointLocation', '/Volumes/workspace/bronze/bronzevolume/bookings/checkpoint')
    # Start the Auto Loader streaming job
    .start())

In [0]:
%sql
-- Checking the loaded data
select * from delta.`/Volumes/workspace/bronze/bronzevolume/bookings/data/`
limit 10

booking_id,passenger_id,flight_id,airport_id,amount,booking_date,_rescued_data
B01001,P0207,F0047,A004,912.64,2025-07-10,null
B01002,P0124,F0088,A051,597.04,2025-07-01,null
B01003,P0103,F0096,A036,724.88,2025-07-10,null
B01004,P0049,F0069,A001,1370.99,2025-07-12,null
B01005,P0210,F0084,A053,1476.15,2025-07-01,null
B01006,P0126,F0084,A042,800.39,2025-06-26,null
B01007,P0077,F0013,A055,1187.22,2025-06-28,null
B01008,P0118,F0005,A033,913.43,2025-07-18,null
B01009,P0090,F0102,A034,260.75,2025-07-08,null
B01010,P0012,F0080,A045,245.61,2025-07-21,null


Uploaded the incremented booking data into raw volume

In [0]:
# Writing the new incremented booking data into Delta Lake (bronze layer)
df.writeStream.format('delta')\
.outputMode('append')\
.trigger(once=True)\
.option('path', '/Volumes/workspace/bronze/bronzevolume/bookings/data')\
.option('checkpointLocation', '/Volumes/workspace/bronze/bronzevolume/bookings/checkpoint')\
.start()



In [0]:
%sql
-- Check if the bronze booking table is incremented
select * from delta.`/Volumes/workspace/bronze/bronzevolume/bookings/data/`
limit 10

booking_id,passenger_id,flight_id,airport_id,amount,booking_date,_rescued_data
B01001,P0207,F0047,A004,912.64,2025-07-10,null
B01002,P0124,F0088,A051,597.04,2025-07-01,null
B01003,P0103,F0096,A036,724.88,2025-07-10,null
B01004,P0049,F0069,A001,1370.99,2025-07-12,null
B01005,P0210,F0084,A053,1476.15,2025-07-01,null
B01006,P0126,F0084,A042,800.39,2025-06-26,null
B01007,P0077,F0013,A055,1187.22,2025-06-28,null
B01008,P0118,F0005,A033,913.43,2025-07-18,null
B01009,P0090,F0102,A034,260.75,2025-07-08,null
B01010,P0012,F0080,A045,245.61,2025-07-21,null


We observed additional newly ingested rows (300 rows appended in this run) \
Shows that autoloader is working fine

However this static process needs to be done for all other files too. So instead we will use a dynamic approach. We will generate a widget box whose input will ingest the folder data into bronze layer

In [0]:
# Create a widget box with name src
dbutils.widgets.text("src", "")

# Display the widget box's inputted value
src_value = dbutils.widgets.get("src")
src_value

''

In [0]:
df = (spark.readStream.format('cloudfiles')
    .option('cloudfiles.format', 'csv')
    # Using dynamic value from the widget box
    .option('cloudfiles.schemaLocation', f'/Volumes/workspace/bronze/bronzevolume/{src_value}/checkpoint')
    .option('cloudfiles.schemaEvolutionMode', 'rescue')
    .load(f'/Volumes/workspace/raw/rawvolume/rawdata/{src_value}/'))

# Writing the inputted value's (airports) data into bronze layer
df.writeStream.format('delta')\
.outputMode('append')\
.trigger(once=True)\
.option('path', f'/Volumes/workspace/bronze/bronzevolume/{src_value}/data')\
.option('checkpointLocation', f'/Volumes/workspace/bronze/bronzevolume/{src_value}/checkpoint')\
.start()    

In [0]:
%sql
-- Check if airports data is ingested into the bronze booking table 
select * from delta.`/Volumes/workspace/bronze/bronzevolume/airports/data/`
limit 10

airport_id,airport_name,city,country,_rescued_data
A023,Maryfurt International Airport,Jessicafurt,Yemen,null
A039,Reneeborough International Airport,New Larryfurt,Saint Pierre and Miquelon,null
A006,Michaelburgh International Airport,Port Josephmouth,Turks and Caicos Islands,null
A056,West Shirleyfort International Airport,West Joshua,Somalia,null
A057,Lake Keith International Airport,Matthewland,Nicaragua,null
A058,Ronniefurt International Airport,Smithhaven,Gabon,null
A051,Amandaport International Airport,Jamestown,Micronesia,null
A052,East Cynthia International Airport,Gibbsfurt,Ecuador,null
A053,South Corey International Airport,Johnsonfort,Switzerland,null
A054,Jonestown International Airport,Garciaview,Colombia,null


We observed that the airports data was ingested successfully. However this process is still static as we need to manually type the value. Instead we will create another notebook which has source parameters to pass in here. 